In [ ]:
import pandas as pd
import random
import numpy as np

In [2]:
Dropbox_path='~/NYU Langone Health Dropbox/Bianca Cordazzo Vargas/Shenhav_Lab/IMiC'
misame_data_path=f'{Dropbox_path}/Data/MISAME'
vital_data_path=f'{Dropbox_path}/Data/VITAL'
sapient_data_path=f'{Dropbox_path}/Data/Sapient_Box_Data/Raw'
vital_untargeted_out=f'{Dropbox_path}/Code/Joint-RPCA/output/all_tps'
metab_id_path=f'{Dropbox_path}/Code/machine_learning'

In [3]:
print("loading metadata")
vital_full_BM_df_filt=pd.read_csv(f"{vital_data_path}/metadata/vital_processed_metadata.tsv", sep='\t')

loading metadata


In [4]:
print("loading data")
vital_raw = pd.read_csv(f"{sapient_data_path}/CHILD_ELICIT_VITAL_rLC_mtb_raw_data_021224.csv", sep=',', index_col=0)
vital_raw = vital_raw[vital_raw.index.isin(vital_full_BM_df_filt['SampleID'])]
vital_array = vital_raw.values
random.seed(100)
#draw from unif distribution with low=min(table)/10 and high=min(table)
zero_len = len(vital_array[pd.isna(vital_array)])
min_value = np.nanmin(vital_array)
rand_array = np.random.uniform(low=min_value/10, high=min_value, size=zero_len)
vital_array[pd.isna(vital_array)] = rand_array
vital_preproc=pd.DataFrame(vital_array, index=vital_raw.index, columns=vital_raw.columns).reset_index()
print(vital_preproc.shape)

loading data
(300, 60609)


In [5]:
print("reading keys")
# Subsetting the original key: remove NAs, sort by rt_difference, remove duplicates from misame and vital ID columns with higher rt_difference 
cross_study_key_original_besthits=pd.read_csv(f"{misame_data_path}/key/IMiC_alignment.csv", sep=',')
cross_study_key_original_besthits=cross_study_key_original_besthits.dropna(subset=['mtb_id_MISAME3', 
                                                                                   'mtb_id_CHILD_ELICIT_VITAL']).sort_values(by='rt_difference')[['mtb_id_MISAME3', 
                                                                                                                                                  'mtb_id_CHILD_ELICIT_VITAL']]
cross_study_key_original_besthits=cross_study_key_original_besthits.drop_duplicates(subset='mtb_id_MISAME3', keep="first")
cross_study_key_original_besthits=cross_study_key_original_besthits.drop_duplicates(subset='mtb_id_CHILD_ELICIT_VITAL', keep="first")
cross_study_key_original_besthits

# Identifying 1-1 overlap IDs from original key
Vital_MisVit2_common_IDs=cross_study_key_original_besthits['mtb_id_CHILD_ELICIT_VITAL'].tolist()
MisVit2_rename_dict=dict(zip(cross_study_key_original_besthits['mtb_id_CHILD_ELICIT_VITAL'], cross_study_key_original_besthits['mtb_id_MISAME3']))
vital_1to1_preproc=vital_preproc[['sample_ID'] + Vital_MisVit2_common_IDs].rename(columns=MisVit2_rename_dict)

# save
#vital_1to1_preproc.to_csv(f"{vital_untargeted_out}/vital_untargeted_all.csv", index=False)

reading keys


In [6]:
#lead top 104 untargeted metabolites (optional)
top_untargeted_metab = pd.read_excel(f'{metab_id_path}/IMiC_Azad_Metabolite_ID_Share.xlsx', index_col=0)
#keep only the top 104 untargeted metabolites
vital_1to1_preproc.set_index('sample_ID', inplace=True)
untargeted = vital_1to1_preproc[top_untargeted_metab.index]

In [7]:
# save
#untargeted.to_csv(f"{vital_untargeted_out}/vital_untargeted_sapient.csv", index=True)